# B2-024-gpu-scientific-ml-capstone — Practice p03 — Solution

**Type:** mc · **Difficulty:** intro · **Concepts:** open-ended-experiment-design

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

The hypothesis names one change (Gaussian input-noise augmentation, std $0.03$ on the log magnitudes), one metric and direction (mean **validation** position error goes down), a minimum effect (at least 5% relative), and a protocol (R's architecture, preprocessing, 300 full-batch Adam steps at $3\times10^{-3}$, seeds 0, 1, 2).
A test of it must compare R with R-plus-augmentation **under that protocol**, changing nothing else, on the **same seeds**, using **validation** numbers only (Session 2, §§1, 3, 5, 6).

Option **B** does exactly this: the baseline is R itself (the strong internal baseline), the ablation changes only the augmentation, the seeds and budget are shared so the three per-seed differences are paired, and the decision uses the mean validation error (checked against the 5% bar) and the per-seed paired differences (to see whether the change exceeds seed noise).

**Answer: B.**

**Rejected options and the principle each violates.**

* **A** — *comparison under the same protocol* and *held-out discipline*: grid Tikhonov is a different method with a different protocol, so the comparison says nothing about augmentation on R; and it compares `final_test_score` results, using the locked test set for a design decision.
* **C** — *one change at a time*: width 128 and 600 steps are changed together with the augmentation, so any difference is confounded among three factors (and the step budget is no longer equal).
* **D** — *shared seeds*: the arms use different seeds (0 versus 1), so the difference mixes the augmentation effect with seed-to-seed noise, and one seed per arm gives no estimate of that noise.
* **E** — *held-out discipline*: choosing between configurations by test score turns the locked test set into a selection set (two test calls, optimistic final number); selection belongs on validation.

In [ ]:
OPTIONS = {
    "A": {"same_protocol": False, "one_change": True, "shared_seeds": True, "validation_only": False},
    "B": {"same_protocol": True, "one_change": True, "shared_seeds": True, "validation_only": True},
    "C": {"same_protocol": True, "one_change": False, "shared_seeds": True, "validation_only": True},
    "D": {"same_protocol": True, "one_change": True, "shared_seeds": False, "validation_only": True},
    "E": {"same_protocol": True, "one_change": True, "shared_seeds": True, "validation_only": False},
}
valid = [k for k, v in OPTIONS.items() if all(v.values())]
violations = {k: [p for p, ok in v.items() if not ok] for k, v in OPTIONS.items() if k not in valid}
ANSWER = valid[0]
print("valid design:", valid, "| violations:", violations)

# The 5% decision bar, applied to a hypothetical R mean of 0.0120:
r_mean = 0.0120
threshold = 0.95 * r_mean
print("augmentation is adopted only if its mean validation error <= %.5f" % threshold)

### Answer check

In [ ]:
assert valid == ["B"] and ANSWER == "B"
assert violations["A"] == ["same_protocol", "validation_only"]
assert violations["C"] == ["one_change"]
assert violations["D"] == ["shared_seeds"]
assert violations["E"] == ["validation_only"]
assert abs(threshold - 0.0114) < 1e-12